In [1]:
!git clone https://github.com/AI4Finance-Foundation/FinGPT.git

Cloning into 'FinGPT'...
remote: Enumerating objects: 2489, done.
remote: Counting objects: 100% (781/781), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 2489 (delta 579), reused 571 (delta 521), pack-reused 1708
Receiving objects: 100% (2489/2489), 10.04 MiB | 16.64 MiB/s, done.
Resolving deltas: 100% (1518/1518), done.


In [2]:
import os
os.chdir('/content/FinGPT/fingpt/FinGPT_Forecaster')

In [ ]:
!pip install -r /content/FinGPT/fingpt/FinGPT_Forecaster/requirements.txt

In [21]:
!pip install datasets
!pip install rouge_score
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.4/221.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llmx 0.0.15a0 requires cohere, which is not installed.
llmx 0.0.15a0 requires tiktoken, which is not installed.


# Prepare Data

In [22]:
import os
import re
import csv
import math
import time
import json
import random
import finnhub
import datasets
import pandas as pd
import yfinance as yf
from datetime import datetime
from collections import defaultdict
from datasets import Dataset
from openai import OpenAI

In [23]:
START_DATE = "2022-12-31"
END_DATE = "2023-05-31"

DATA_DIR = f"./{START_DATE}_{END_DATE}"
os.makedirs(DATA_DIR, exist_ok=True)

finnhub_client = finnhub.Client(api_key="")

client = OpenAI(api_key = '')

In [24]:
def bin_mapping(ret):

    up_down = 'U' if ret >= 0 else 'D'
    integer = math.ceil(abs(100 * ret))

    return up_down + (str(integer) if integer <= 5 else '5+')


def get_returns(stock_symbol):

    # Download historical stock data
    stock_data = yf.download(stock_symbol, start=START_DATE, end=END_DATE)

    weekly_data = stock_data['Adj Close'].resample('W').ffill()
    weekly_returns = weekly_data.pct_change()[1:]
    weekly_start_prices = weekly_data[:-1]
    weekly_end_prices = weekly_data[1:]

    weekly_data = pd.DataFrame({
        'Start Date': weekly_start_prices.index,
        'Start Price': weekly_start_prices.values,
        'End Date': weekly_end_prices.index,
        'End Price': weekly_end_prices.values,
        'Weekly Returns': weekly_returns.values
    })

    weekly_data['Bin Label'] = weekly_data['Weekly Returns'].map(bin_mapping)

    return weekly_data


def get_news(symbol, data):

    news_list = []

    for end_date, row in data.iterrows():
        start_date = row['Start Date'].strftime('%Y-%m-%d')
        end_date = row['End Date'].strftime('%Y-%m-%d')
        print(symbol, ': ', start_date, ' - ', end_date)
        time.sleep(1) # control qpm
        weekly_news = finnhub_client.company_news(symbol, _from=start_date, to=end_date)
        weekly_news = [
            {
                "date": datetime.fromtimestamp(n['datetime']).strftime('%Y%m%d%H%M%S'),
                "headline": n['headline'],
                "summary": n['summary'],
            } for n in weekly_news
        ]
        weekly_news.sort(key=lambda x: x['date'])
        news_list.append(json.dumps(weekly_news))

    data['News'] = news_list

    return data


def get_basics(symbol, data, always=False):

    basic_financials = finnhub_client.company_basic_financials(symbol, 'all')

    final_basics, basic_list, basic_dict = [], [], defaultdict(dict)

    for metric, value_list in basic_financials['series']['quarterly'].items():
        for value in value_list:
            basic_dict[value['period']].update({metric: value['v']})

    for k, v in basic_dict.items():
        v.update({'period': k})
        basic_list.append(v)

    basic_list.sort(key=lambda x: x['period'])

    for i, row in data.iterrows():

        start_date = row['End Date'].strftime('%Y-%m-%d')
        last_start_date = START_DATE if i < 2 else data.loc[i-2, 'Start Date'].strftime('%Y-%m-%d')

        used_basic = {}
        for basic in basic_list[::-1]:
            if (always and basic['period'] < start_date) or (last_start_date <= basic['period'] < start_date):
                used_basic = basic
                break
        final_basics.append(json.dumps(used_basic))

    data['Basics'] = final_basics

    return data


def prepare_data_for_company(symbol, with_basics=True):

    data = get_returns(symbol)
    data = get_news(symbol, data)

    if with_basics:
        data = get_basics(symbol, data)
        data.to_csv(f"{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}.csv")
    else:
        data['Basics'] = [json.dumps({})] * len(data)
        data.to_csv(f"{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_nobasics.csv")

    return data

In [25]:
DOW_30 = [
    "AXP", "AMGN", "AAPL", "BA", "CAT", "CSCO", "CVX", "GS", "HD", "HON",
    "IBM", "INTC", "JNJ", "KO", "JPM", "MCD", "MMM", "MRK", "MSFT", "NKE",
    "PG", "TRV", "UNH", "CRM", "VZ", "V", "WBA", "WMT", "DIS", "DOW"
]

In [26]:
for symbol in DOW_30:
    prepare_data_for_company(symbol)

[*********************100%%**********************]  1 of 1 completed
AXP :  2023-01-08  -  2023-01-15
AXP :  2023-01-15  -  2023-01-22
AXP :  2023-01-22  -  2023-01-29
AXP :  2023-01-29  -  2023-02-05
AXP :  2023-02-05  -  2023-02-12
AXP :  2023-02-12  -  2023-02-19
AXP :  2023-02-19  -  2023-02-26
AXP :  2023-02-26  -  2023-03-05
AXP :  2023-03-05  -  2023-03-12
AXP :  2023-03-12  -  2023-03-19
AXP :  2023-03-19  -  2023-03-26
AXP :  2023-03-26  -  2023-04-02
AXP :  2023-04-02  -  2023-04-09
AXP :  2023-04-09  -  2023-04-16
AXP :  2023-04-16  -  2023-04-23
AXP :  2023-04-23  -  2023-04-30
AXP :  2023-04-30  -  2023-05-07
AXP :  2023-05-07  -  2023-05-14
AXP :  2023-05-14  -  2023-05-21
AXP :  2023-05-21  -  2023-05-28
AXP :  2023-05-28  -  2023-06-04
[*********************100%%**********************]  1 of 1 completed
AMGN :  2023-01-08  -  2023-01-15
AMGN :  2023-01-15  -  2023-01-22
AMGN :  2023-01-22  -  2023-01-29
AMGN :  2023-01-29  -  2023-02-05
AMGN :  2023-02-05  -  2023-02-12

In [27]:
def get_company_prompt(symbol):

    profile = finnhub_client.company_profile2(symbol=symbol)

    company_template = "[Company Introduction]:\n\n{name} is a leading entity in the {finnhubIndustry} sector. Incorporated and publicly traded since {ipo}, the company has established its reputation as one of the key players in the market. As of today, {name} has a market capitalization of {marketCapitalization:.2f} in {currency}, with {shareOutstanding:.2f} shares outstanding." \
        "\n\n{name} operates primarily in the {country}, trading under the ticker {ticker} on the {exchange}. As a dominant force in the {finnhubIndustry} space, the company continues to innovate and drive progress within the industry."

    formatted_str = company_template.format(**profile)

    return formatted_str


def get_prompt_by_row(symbol, row):

    start_date = row['Start Date'] if isinstance(row['Start Date'], str) else row['Start Date'].strftime('%Y-%m-%d')
    end_date = row['End Date'] if isinstance(row['End Date'], str) else row['End Date'].strftime('%Y-%m-%d')
    term = 'increased' if row['End Price'] > row['Start Price'] else 'decreased'
    head = "From {} to {}, {}'s stock price {} from {:.2f} to {:.2f}. Company news during this period are listed below:\n\n".format(
        start_date, end_date, symbol, term, row['Start Price'], row['End Price'])

    news = json.loads(row["News"])
    news = ["[Headline]: {}\n[Summary]: {}\n".format(
        n['headline'], n['summary']) for n in news if n['date'][:8] <= end_date.replace('-', '') and \
        not n['summary'].startswith("Looking for stock market analysis and research with proves results?")]

    basics = json.loads(row['Basics'])
    if basics:
        basics = "Some recent basic financials of {}, reported at {}, are presented below:\n\n[Basic Financials]:\n\n".format(
            symbol, basics['period']) + "\n".join(f"{k}: {v}" for k, v in basics.items() if k != 'period')
    else:
        basics = "[Basic Financials]:\n\nNo basic financial reported."

    return head, news, basics


def sample_news(news, k=5):

    return [news[i] for i in sorted(random.sample(range(len(news)), k))]


def map_bin_label(bin_lb):

    lb = bin_lb.replace('U', 'up by ')
    lb = lb.replace('D', 'down by ')
    lb = lb.replace('1', '0-1%')
    lb = lb.replace('2', '1-2%')
    lb = lb.replace('3', '2-3%')
    lb = lb.replace('4', '3-4%')
    if lb.endswith('+'):
        lb = lb.replace('5', 'more than 5%')
#         lb = lb.replace('5+', '5+%')
    else:
        lb = lb.replace('5', '4-5%')

    return lb


def get_all_prompts(symbol, min_past_weeks=1, max_past_weeks=3, with_basics=True):


    if with_basics:
        df = pd.read_csv(f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}.csv')
    else:
        df = pd.read_csv(f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_nobasics.csv')

    company_prompt = get_company_prompt(symbol)

    prev_rows = []
    all_prompts = []

    for row_idx, row in df.iterrows():

        prompt = ""
        if len(prev_rows) >= min_past_weeks:
            idx = min(random.choice(range(min_past_weeks, max_past_weeks+1)), len(prev_rows))
            for i in range(-idx, 0):
                # Add Price Movement (Head)
                prompt += "\n" + prev_rows[i][0]
                # Add News of previous weeks
                sampled_news = sample_news(
                    prev_rows[i][1],
                    min(5, len(prev_rows[i][1]))
                )
                if sampled_news:
                    prompt += "\n".join(sampled_news)
                else:
                    prompt += "No relative news reported."

        head, news, basics = get_prompt_by_row(symbol, row)

        prev_rows.append((head, news, basics))
        if len(prev_rows) > max_past_weeks:
            prev_rows.pop(0)

        if not prompt:
            continue

        prediction = map_bin_label(row['Bin Label'])

        prompt = company_prompt + '\n' + prompt + '\n' + basics
        prompt += f"\n\nBased on all the information before {row['Start Date']}, let's first analyze the positive developments and potential concerns for {symbol}. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company related news. " \
            f"Then let's assume your prediction for next week ({row['Start Date']} to {row['End Date']}) is {prediction}. Provide a summary analysis to support your prediction. The prediction result need to be inferred from your analysis at the end, and thus not appearing as a foundational factor of your analysis."

        all_prompts.append(prompt.strip())

    return all_prompts

In [28]:
B_INST, E_INST = "[INST]", "[/INST]"
B_SYS, E_SYS = "<<SYS>>\n", "\n<</SYS>>\n\n"


SYSTEM_PROMPT = "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. " \
    "Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]:\n...\n"

print(SYSTEM_PROMPT)

# prompts = get_all_prompts("AAPL", 1, 3)
# prompts = get_all_prompts("MSFT", 1, 3, False)
prompts = get_all_prompts("TRV", 1, 4)

print(prompts[0])

You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:

[Positive Developments]:
1. ...

[Potential Concerns]:
1. ...

[Prediction & Analysis]:
...

[Company Introduction]:

Travelers Companies Inc is a leading entity in the Insurance sector. Incorporated and publicly traded since 1980-03-17, the company has established its reputation as one of the key players in the market. As of today, Travelers Companies Inc has a market capitalization of 41947.85 in USD, with 228.40 shares outstanding.

Travelers Companies Inc operates primarily in the US, trading under the ticker TRV on the NEW YORK STOCK EXCHANGE, INC.. As a dominant force in the Insurance space, the company continues to innovate and drive progress within 

# Request to GPT-4 for Financial Analysis

In [29]:
def append_to_csv(filename, input_data, output_data):

    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([input_data, output_data])


def initialize_csv(filename):

    with open(filename, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["prompt", "answer"])


def query_gpt4(symbol_list, min_past_weeks=1, max_past_weeks=3, with_basics=True):

    for symbol in symbol_list:

        csv_file = f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_gpt-4.csv' if with_basics else \
                   f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_nobasics_gpt-4.csv'

        if not os.path.exists(csv_file):
            initialize_csv(csv_file)
            pre_done = 0
        else:
            df = pd.read_csv(csv_file)
            pre_done = len(df)

        prompts = get_all_prompts(symbol, min_past_weeks, max_past_weeks, with_basics)

        for i, prompt in enumerate(prompts):

            if i < pre_done:
                continue

            print(f"{symbol} - {i}")

            cnt = 0
            while cnt < 5:
                try:
                    completion = client.chat.completions.create(
                        model="gpt-4",
                        messages=[
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": prompt}
                          ]
                    )
                    break
                except Exception:
                    cnt += 1
                    print(f'retry cnt {cnt}')

            answer = completion.choices[0].message.content if cnt < 5 else ""
            append_to_csv(csv_file, prompt, answer)

In [ ]:
query_gpt4(DOW_30, 1, 4)

AXP - 0
AXP - 1
AXP - 2
AXP - 3
AXP - 4
AXP - 5
AXP - 6
AXP - 7
AXP - 8
AXP - 9


# Transform into Llama2 Training Format

In [ ]:
def gpt4_to_llama(symbol, with_basics=True):

    csv_file = f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_gpt-4.csv' if with_basics else \
                   f'{DATA_DIR}/{symbol}_{START_DATE}_{END_DATE}_nobasics_gpt-4.csv'

    df = pd.read_csv(csv_file)

    prompts, answers, periods, labels = [], [], [], []

    for i, row in df.iterrows():

        prompt, answer = row['prompt'], row['answer']

        res = re.search(r"Then let's assume your prediction for next week \((.*)\) is ((:?up|down) by .*%).", prompt)

        period, label = res.group(1), res.group(2)
#         label = label.replace('more than 5', '5+')

        prompt = re.sub(
            r"Then let's assume your prediction for next week \((.*)\) is (up|down) by ((:?.*)%). Provide a summary analysis to support your prediction. The prediction result need to be inferred from your analysis at the end, and thus not appearing as a foundational factor of your analysis.",
            f"Then make your prediction of the {symbol} stock price movement for next week ({period}). Provide a summary analysis to support your prediction.",
            prompt
        )
        try:
            answer = re.sub(
                r"\[Prediction & Analysis\]:\s*",
                f"[Prediction & Analysis]:\nPrediction: {label.capitalize()}\nAnalysis: ",
                answer
            )
        except Exception:
            print(symbol, i)
            print(label)
            print(answer)
            continue

        new_system_prompt = SYSTEM_PROMPT.replace(':\n...', '\nPrediction: ...\nAnalysis: ...')
#         new_system_prompt = SYSTEM_PROMPT.replace(':\n...', '\nPrediction: {Up|Down} by {1-2|2-3|3-4|4-5|5+}%\nAnalysis: ...')

        prompt = B_INST + B_SYS + new_system_prompt + E_SYS + prompt + E_INST

        prompts.append(prompt)
        answers.append(answer)
        periods.append(period)
        labels.append(label)

    return {
        "prompt": prompts,
        "answer": answers,
        "period": periods,
        "label": labels,
    }


def create_dataset(symbol_list, train_ratio=0.8, with_basics=True):

    train_dataset_list = []
    test_dataset_list = []

    for symbol in symbol_list:

        data_dict = gpt4_to_llama(symbol, with_basics)
#         print(data_dict['prompt'][-1])
#         print(data_dict['answer'][-1])
        symbols = [symbol] * len(data_dict['label'])
        data_dict.update({"symbol": symbols})

        dataset = Dataset.from_dict(data_dict)
        train_size = round(train_ratio * len(dataset))

        train_dataset_list.append(dataset.select(range(train_size)))
        test_dataset_list.append(dataset.select(range(train_size, len(dataset))))

    train_dataset = datasets.concatenate_datasets(train_dataset_list)
    test_dataset = datasets.concatenate_datasets(test_dataset_list)

    dataset = datasets.DatasetDict({
        'train': train_dataset,
        'test': test_dataset
    })

    return dataset


In [ ]:
dow30_v3_dataset = create_dataset(DOW_30, 0.9)

In [ ]:
dow30_v3_dataset.save_to_disk('data/fingpt-forecaster-dow30v3-20221231-20230531-llama')

# Load model

In [11]:
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from utils import *

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    'NousResearch/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model.model_parellal = True

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:362: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:367: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(


In [14]:
model = PeftModel.from_pretrained(base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora')
model = model.eval()

adapter_config.json:   0%|          | 0.00/528 [00:00<?, ?B/s]

adapter_model.bin:   0%|          | 0.00/40.1M [00:00<?, ?B/s]

In [16]:
tokenizer = AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-chat-hf')
tokenizer.padding_side = "right"
tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [17]:
test_dataset = load_from_disk('data/fingpt-forecaster-dow30v3-20221231-20230531-llama/')['test']

FileNotFoundError: ignored

In [18]:
def test_demo(model, tokenizer, prompt):

    inputs = tokenizer(
        prompt, return_tensors='pt',
        padding=False, max_length=4096
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    res = model.generate(
        **inputs, max_length=4096, do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True
    )
    output = tokenizer.decode(res[0], skip_special_tokens=True)
    return output
    # return res

In [19]:
answers, gts = [], []

for i in range(len(test_dataset)):
    prompt = test_dataset[i]['prompt']
    output = test_demo(model, tokenizer, prompt)
    answer = re.sub(r'.*\[/INST\]\s*', '', output, flags=re.DOTALL)
    gt = test_dataset[i]['answer']
    print('\n------- Prompt ------\n')
    print(prompt)
    print('\n------- LLaMA2 Finetuned ------\n')
    print(answer)
    print('\n------- GPT4 Groundtruth ------\n')
    print(gt)
    print('\n===============\n')
    answers.append(answer)
    gts.append(gt)

NameError: ignored

In [ ]:
gts[0]

"[Positive Developments]:\n1. The declaration of a regular quarterly dividend, amounting to $0.60 per common share, indicates a steady cash return to shareholders. This can boost investor confidence, leading to increased demand for AXP shares.\n2. A company executive's participation in major industry conferences offers the chance to elaborate on American Express's growth strategy and financial performance. Such events could provide positive exposure for the company, potentially driving stock price gains.\n3. The article highlighting American Express's double-digit revenue growth and aggressive share buyback strategy illustrates strong company performance, a factor that can lead to increased investor interest.\n\n[Potential Concerns]:\n1. The interest rate hike by the Federal Reserve could put pressure on financial institutions, affecting their loans' profitability and potentially leading to a decline in stock prices.\n2. According to one report, concerns exist that credit card profitab

In [ ]:
answers[0]

"[Positive Developments]:\n1. The company's Vice Chairman, Douglas E. Buckminster, will participate in the SVB MoffettNathanson Technology, Media, and Telecom Conference, which could potentially increase investor confidence and interest in the company.\n2. American Express declared a regular quarterly dividend, which may attract dividend-focused investors.\n3. The company's stock price increased from 146.82 to 151.80 in the week from 2023-05-14 to 2023-05-21, indicating potential upward trends.\n\n[Potential Concerns]:\n1. The company's stock price has been declining since the beginning of May, which may indicate a potential downward trend.\n2. There have been concerns about the company's profitability after a strong 2022, which could indicate a potential peak in profitability.\n3. The company's stock has been mentioned in articles discussing the potential risks of fintech companies and the regional banking crisis, which could impact the company's profitability and stock price.\n\n[Pre

In [ ]:
calc_metrics(answers, gts)


Binary Accuracy: 0.61  |  Mean Square Error: 5.93

Rouge Score of Positive Developments: {'rouge1': 0.43196333904653406, 'rouge2': 0.15722817931423375, 'rougeL': 0.264313106777193}

Rouge Score of Potential Concerns: {'rouge1': 0.39554620999107204, 'rouge2': 0.12632646861001814, 'rougeL': 0.24735208697254654}

Rouge Score of Summary Analysis: {'rouge1': 0.42928213837028545, 'rouge2': 0.1354372841499895, 'rougeL': 0.2320299101133297}


{'valid_count': 57,
 'bin_acc': 0.6140350877192983,
 'mse': 5.9298245614035086,
 'pros_rouge_scores': {'rouge1': 0.43196333904653406,
  'rouge2': 0.15722817931423375,
  'rougeL': 0.264313106777193},
 'cons_rouge_scores': {'rouge1': 0.39554620999107204,
  'rouge2': 0.12632646861001814,
  'rougeL': 0.24735208697254654},
 'anal_rouge_scores': {'rouge1': 0.42928213837028545,
  'rouge2': 0.1354372841499895,
  'rougeL': 0.2320299101133297}}